<a href="https://colab.research.google.com/github/Apurva-J509/Linear_Regression/blob/main/Multiple_Linear_Regression_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [8]:
students = pd.read_csv('students_dataset.csv')

In [9]:
X = students.loc[:, ['CGPA', 'IQ']]
y = students.loc[:, ['Package_LPA']]

### Multiple Linear Regression

In [10]:
#Splitting dataset into X_train, X_test, y_train and y_test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=42)

In [40]:
#Multiple Linear Regression class from scratch using closed form solution
class MultipleLinearRegression:
  def __init__(self):
    self.coef_ = None
    self.intercept = None
  def fit(self, X_train, y_train):
    X_train = np.insert(X_train, 0, 1, axis=1)
    theta = np.linalg.inv(X_train.T.dot(X_train)).dot(X_train.T).dot(y_train)
    self.intercept_ = theta[0]
    self.coef_ = theta[1:]
  def predict(self, X_test):
    X_test = np.insert(X_test, 0, 1, axis=1)
    return X_test.dot(np.concatenate((np.expand_dims(regressor.intercept_, axis = 0), regressor.coef_), axis = 0))


In [41]:
regressor = MultipleLinearRegression()

In [42]:
#Training model on train dataset
regressor.fit(X_train, y_train)

In [81]:
print('Weights of predictors: \n', regressor.coef_)
print('Intercept: ', regressor.intercept_)

Weights of predictors: 
 [[3.15470481]
 [0.35548236]]
Intercept:  [-49.73168391]


In [82]:
#Predicting package of test dataset
y_predicted = regressor.predict(X_test)

### Accuracy of Model

In [69]:
#Residual Sum of Squares
def rss(y_predicted, y_test):
  y_test = np.array(y_test)
  return np.sum((y_predicted - y_test)**2)
#Total Sum of Squares
def tss(y_predicted, y_test):
  y_test = np.array(y_test)
  return np.sum((y_predicted - y_test.mean())**2)
#Residual Standard Error
def rse(y_predicted, y_test, no_predictors):
  y_test = np.array(y_test)
  df = len(y_predicted) - no_predictors -1
  return np.sqrt(rss(y_predicted, y_test)/df)
#R2 Score
def r2_score(y_predicted, y_test):
  y_test = np.array(y_test)
  return 1 - (rss(y_predicted, y_test)/tss(y_predicted, y_test))

In [73]:
print("RSE for model: ", rse(y_predicted, y_test, X_train.shape[1]))
print("R2 Score for model: ", r2_score(y_predicted, y_test))

RSE for model:  1.027034997681674
R2 Score for model:  0.9820851080485316


### F-test for Multiple Linear Regression
$$H_{0} = \beta_{0} = \beta_{1} =\beta_{2} = 0$$
$$H_{1} = \beta_{0} \neq \beta_{1} \neq \beta_{2} \neq 0$$

In [78]:
from scipy.stats import f
def f_test(y_predicted, y_test, no_predictors):
  df1 = no_predictors
  df2 = len(y_predicted) - no_predictors -1
  TSS = tss(y_predicted, y_test)
  RSS = rss(y_predicted, y_test)
  f_stat = ((TSS-RSS)/df1)/((RSS)/df2)
  # Calculate lower tail probability (CDF) and upper tail probability (Survival function / 1 - CDF)
  lower_tail = f.cdf(f_stat, df1, df2)
  upper_tail = f.sf(f_stat, df1, df2)

  # For a two-tailed test, double the smaller tail probability
  p_value = 2 * min(lower_tail, upper_tail)
  return  p_value

In [79]:
f_test(y_predicted, y_test, X_train.shape[1])

np.float64(5.241089833736568e-24)